# Mesa de Ayuda IA — demo paso a paso

Un estudio contable recibe todos los días consultas de facturación de un cliente. Muchas **ya se respondieron**
(a otra persona del mismo cliente, o a la misma persona hace unos meses). Este cuaderno muestra cómo el asistente:

1. arma la "memoria" a partir del historial de mails (el que junta el Apps Script),
2. busca antecedentes para cada consulta nueva,
3. la clasifica (**Repetida / Posible antecedente / Nueva**) y detecta desorden operativo,
4. propone un borrador que **siempre revisa una persona**.

> Datos ficticios, generados para demostración. Recreación de una herramienta real.

In [1]:
# En Google Colab: clonar el repositorio (en local no hace falta)
import os, sys
if "google.colab" in sys.modules and not os.path.exists("mesa-de-ayuda-ia"):
    !git clone -q https://github.com/ClaudioCGG/mesa-de-ayuda-ia.git
    os.chdir("mesa-de-ayuda-ia")
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, "src")

import pandas as pd
pd.set_option("display.max_colwidth", 90)

## 1. El historial: lo que ya se respondió

In [2]:
from mesa_ayuda.historial import cargar_mensajes, construir_casos

mensajes = cargar_mensajes("data/historial.csv")
casos = construir_casos(mensajes)
print(f"{len(mensajes)} mensajes · {mensajes['Hilo ID'].nunique()} hilos · {len(casos)} casos respondidos")
casos[["fecha", "persona", "asunto", "pregunta", "respuesta"]].head()

153 mensajes · 64 hilos · 64 casos respondidos


,fecha,persona,asunto,pregunta,respuesta
0,2026-05-20 17:54:00,admin,Factura abril - urgente,"Buenas, necesito la factura del mes de abril, ¿la pueden mandar de nuevo? Gracias, Admin",Las facturas se envían el día 5 de cada mes a la casilla de compras de ustedes y ademá...
1,2026-01-04 10:28:00,carla.benitez,Factura con datos mal,"La factura salió con el CUIT equivocado, ¿cómo la corregimos? Gracias, Carla",Una factura electrónica no se puede modificar una vez emitida. El procedimiento es emi...
2,2026-08-12 11:39:00,mariela.paz,¿Cuándo se paga el IVA?,"¿Cuándo vence el IVA de julio? Gracias, Mariela",El IVA vence según la terminación del CUIT: para ustedes (terminación 7) vence el día ...
3,2026-08-04 09:49:00,admin,Les paso el pago,"Adjunto foto del pago que hicimos ayer. Gracias, Admin",Para registrar el pago necesitamos el comprobante en PDF que emite el banco (no una ca...
4,2026-09-05 10:29:00,carla.benitez,Nota de crédito,"Un cliente nos devolvió mercadería, ¿cómo hacemos la nota de crédito? Gracias, Carla",Para emitir la nota de crédito por devolución necesitamos: número de la factura origin...


## 2. ¿Esta consulta ya se respondió?

La búsqueda compara textos con TF-IDF sobre fragmentos de 3 a 5 letras: tolera errores de tipeo y variantes
("reenviar", "reenvían", "reenvío") sin necesitar internet ni API.

In [3]:
from mesa_ayuda.buscador import BuscadorAntecedentes

buscador = BuscadorAntecedentes(casos)
buscador.buscar("hola, me pasan de nuevo la factura del mes pasado? no la encuentro", k=3)[
    ["fecha", "persona", "asunto", "similitud"]]

,fecha,persona,asunto,similitud
0,2026-01-07 15:39:00,mariela.paz,No encuentro la factura,0.687795
1,2026-02-16 14:05:00,carla.benitez,No encuentro la factura,0.621948
2,2026-04-17 16:38:00,mariela.paz,Factura marzo - urgente,0.505451


## 3. La bandeja de la semana

Cada consulta se clasifica según qué tan claro es el antecedente:

| Clasificación | Similitud | Qué hace el asistente |
|---|---|---|
| Repetida | ≥ 0,45 | cita el antecedente y reutiliza la respuesta |
| Posible antecedente | 0,30 – 0,45 | lo muestra, pero decide una persona |
| Nueva | < 0,30 | acuse de recibo, sin inventar una respuesta |

In [4]:
from mesa_ayuda.asistente import Asistente, RedactorPlantillas

bandeja = pd.read_csv("data/bandeja.csv", parse_dates=["Fecha"])
asistente = Asistente(casos, RedactorPlantillas())
resultados = asistente.procesar_bandeja(bandeja)
resultados[["persona", "asunto", "clasificacion", "tono", "similitud", "alertas"]].round(2)

,persona,asunto,clasificacion,tono,similitud,alertas
0,tomas.quiroga,Factura de septiembre,Posible antecedente,A definir por quien revisa,0.42,[]
1,romina.diaz,Factura con datos mal,Repetida,Cordial con antecedente,0.51,[Consulta respondida 6 veces en el último año.]
2,tomas.quiroga,Vencimiento IVA,Repetida,Cordial con antecedente,0.60,[Consulta respondida 8 veces en el último año.]
3,tomas.quiroga,Pedido de NC,Repetida,Cordial con antecedente,0.52,[Consulta respondida 14 veces en el último año.]
4,tomas.quiroga,Exportación,Repetida,Cordial con antecedente,0.55,[Consulta respondida 13 veces en el último año.]
5,romina.diaz,Consulta tipo de factura,Repetida,Cordial con antecedente,0.79,[Consulta respondida 8 veces en el último año.]
6,jonatan.ruiz,Pago proveedor,Repetida,Firme y pedagógico,0.74,"[Mandó una captura de pantalla en lugar del comprobante en PDF., La misma persona ya h..."
7,admin,URGENTE factura septiembre,Repetida,Firme y pedagógico,0.81,"[La misma persona ya hizo esta consulta el 20/05/2026., Consulta respondida 6 veces en..."
8,lucas.ferro,Facturar un anticipo,Posible antecedente,A definir por quien revisa,0.40,[]
9,carla.benitez,Certificado de retención,Nueva,Cordial,0.17,[]


## 4. Un borrador con desorden operativo (tono firme y pedagógico)

In [5]:
r = resultados[resultados["tono"] == "Firme y pedagógico"].iloc[0]
print("CONSULTA:", r["consulta"]); print("ALERTAS:", *r["alertas"], sep="\n  - "); print()
print(r["borrador"])

CONSULTA: Ahí les mando la captura de pantalla de la transferencia que hicimos hoy.
ALERTAS:
  - Mandó una captura de pantalla en lugar del comprobante en PDF.
  - La misma persona ya hizo esta consulta el 02/06/2026.
  - Consulta respondida 9 veces en el último año.

Hola, Jonatan:

Gracias por enviarlo. Para poder registrarlo necesitamos la documentación completa; te recordamos cómo es el procedimiento:

Para registrar el pago necesitamos el comprobante en PDF que emite el banco (no una captura de pantalla) y la factura del proveedor. Con la captura no podemos verificar el número de operación ni la cuenta de destino.

Para agilizar las próximas gestiones, te sugerimos guardar este mail o compartirlo con el resto del equipo de administración.

Saludos,
Equipo de Facturación


## 5. ¿Qué tan bien funciona?

Como los datos son sintéticos, se conoce la respuesta correcta. La métrica clave es **no equivocarse con
confianza**: una consulta marcada "Repetida" con el antecedente equivocado sería un mail mal respondido.

In [6]:
from mesa_ayuda.metricas import evaluar, resumen

ev = evaluar(resultados, pd.read_csv("data/verdad.csv"), pd.read_csv("data/verdad_historial.csv"))
display(ev[["persona", "tema", "clasificacion", "tema_antecedente", "veredicto"]])
resumen(ev)

,persona,tema,clasificacion,tema_antecedente,veredicto
0,tomas.quiroga,reenvio_factura,Posible antecedente,reenvio_factura,A revisión (antecedente útil)
1,romina.diaz,cuit_erroneo,Repetida,cuit_erroneo,Acierto
2,tomas.quiroga,vencimiento_iva,Repetida,vencimiento_iva,Acierto
3,tomas.quiroga,nota_credito_devolucion,Repetida,nota_credito_devolucion,Acierto
4,tomas.quiroga,factura_exterior,Repetida,factura_exterior,Acierto
5,romina.diaz,tipo_factura,Repetida,tipo_factura,Acierto
6,jonatan.ruiz,comprobante_pago,Repetida,comprobante_pago,Acierto
7,admin,reenvio_factura,Repetida,reenvio_factura,Acierto
8,lucas.ferro,anticipo,Posible antecedente,factura_exterior,A revisión (sin antecedente útil)
9,carla.benitez,certificado_ganancias,Nueva,NaN,Acierto


{'consultas': 11,
 'repetidas_reales': 8,
 'repetidas_con_antecedente_correcto': 8,
 'marcadas_repetida': 7,
 'precision_repetida': 1.0,
 'errores_con_confianza': 0,
 'desorden_reales': 2,
 'desorden_detectados': 2,
 'falsas_alarmas_desorden': 0,
 'a_revision': 2,
 'nuevas_bien_marcadas': 3,
 'nuevas_reales': 3}

## 6. Opcional: borradores con Gemini

Con una API key de Gemini (gratuita en Google AI Studio), los borradores de consultas **repetidas** los redacta
el modelo a partir del antecedente, con las reglas de `prompts/redactor.md`. Las dudosas y las nuevas siguen por
plantilla: al modelo no se le pide que invente cuando no hay antecedente.

```python
# pip install google-genai
import os
os.environ["GEMINI_API_KEY"] = "..."   # en Colab: usar los Secretos
from mesa_ayuda.asistente import crear_redactor
asistente = Asistente(casos, crear_redactor())
```

## 7. La bandeja de revisión (Excel + página)

In [7]:
from mesa_ayuda.salidas import exportar_excel, exportar_html

exportar_excel(resultados, "salida/bandeja_revision.xlsx", ev)
exportar_html(resultados, "salida/bandeja_revision.html", resumen(ev))
print("Listo: salida/bandeja_revision.xlsx y salida/bandeja_revision.html")

Listo: salida/bandeja_revision.xlsx y salida/bandeja_revision.html
